In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy

Matplotlib is building the font cache; this may take a moment.


In [22]:
df = pd.read_csv('../../train.csv', sep=';')
df.head(1)


,medical_specialty,description,sample_name,transcription,keywords
0,Cardiovascular-Pulmonary,"Chest pain, possible syncopal spells. She ha...",Chest Pain - Cardiac Consult,"REASON FOR REFERRAL: Chest pain, possible syn...","cardiovascular / pulmonary, chest pain, syncop..."


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2616 entries, 0 to 2615
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   medical_specialty  2616 non-null   str  
 1   description        2613 non-null   str  
 2   sample_name        2609 non-null   str  
 3   transcription      2583 non-null   str  
 4   keywords           2211 non-null   str  
dtypes: str(5)
memory usage: 102.3 KB


In [24]:
df.describe(include='all')

,medical_specialty,description,sample_name,transcription,keywords
count,2616,2613,2609,2583,2211
unique,15,1818,1833,1816,2171
top,Surgery,Common description of colonoscopy,Lumbar Discogram,"PREOPERATIVE DIAGNOSIS: Low back pain.,POSTOPE...",
freq,876,9,4,4,39


In [25]:
normal_specialties = [s for s in df['medical_specialty'].unique() if len(str(s)) < 30]
print(len(normal_specialties))

12


In [26]:
anomalies = [s for s in df['medical_specialty'] if len(str(s)) > 30]

for x, text in enumerate(anomalies, 1):
    print(f"Anomaly {x} (Length {len(text)}): {text[-200:]}...\n")

Anomaly 1 (Length 3336): ctious Disease consult.  We will see him back in the office early next week to reassess his wound.  He is to keep the wound clean with the moist dressing right now.  He may shower several times a day....

Anomaly 2 (Length 5657): of the cerebral vessels.,2.  Continue Plavix.,3.  Obtain copies of the workup done at the outside hospital.,4.  We will follow the lumbar stenosis for the time being.  No further workup is planned."""...

Anomaly 3 (Length 2067):  with 0.25% Marcaine, and then adjusted with Steri-Strips and bandage appropriately.,The patient was taken from the operating room in stable condition and should be observed overnight in the hospital....



In [27]:
anomaly_rows = df[df['medical_specialty'].str.len() >= 30]

print(f"Indices: {anomaly_rows.index.tolist()}\n")

for idx, row in anomaly_rows.iterrows():
    print("=" * 80)
    print(f"LINHA NO DATASET (Indice {idx}):")
    print("=" * 80)
    print(f"• medical_specialty : {row['medical_specialty'][:150]}... [Tamanho: {len(str(row['medical_specialty']))}]") 
    print(f"• sample_name : {row['sample_name']}") 
    print(f"• description : {row['description']}") 
    print(f"• keywords : {row['keywords']}") 
    print(f"• transcription : {str(row['transcription'])[:150]}...") 
    print("\\n")



Indices: [932, 2052, 2281]

LINHA NO DATASET (Indice 932):
• medical_specialty : d is open about may be roughly a centimeter in the inside of the flap.  He was seen by his medical primary care physician and was given a tetanus shot... [Tamanho: 3336]
• sample_name : nan
• description : nan
• keywords : nan
• transcription : nan...
\n
LINHA NO DATASET (Indice 2052):
• medical_specialty : ypothyroidism secondary to hyperthyroidism and irradiation, possible lumbar stenosis.  He reports he experienced a single episode of his vision decrea... [Tamanho: 5657]
• sample_name : nan
• description : nan
• keywords : nan
• transcription : nan...
\n
LINHA NO DATASET (Indice 2281):
• medical_specialty : ted and this was used to aspirate approximately 30 cc of serosanguineous fluid from the cystic structure.  Next, the needle was removed and the ligatu... [Tamanho: 2067]
• sample_name : nan
• description : nan
• keywords : nan
• transcription : nan...
\n


In [28]:
text_columns = ['description','sample_name','transcription','keywords']
summary = []
for col in text_columns:
    s = df[col].fillna('')

    char_len = s.str.len()
    word_len = s.str.split().str.len()
    
    summary.append({
        'Coluna': col,
        'Valores em Falta(NaN)': df[col].isnull().sum(),
        'Media de chars': round(char_len.mean(),1),
        'Media de Palavras': round(word_len.mean(),1),
        'Min Palavras': word_len.min(),
        'Max Palavras': word_len.max()
    })
summary_df = pd.DataFrame(summary)
display(summary_df)

,Coluna,Valores em Falta(NaN),Media de chars,Media de Palavras,Min Palavras,Max Palavras
0,description,3,200.8,28.4,0,4933
1,sample_name,7,27.1,3.8,0,43
2,transcription,33,2987.3,451.6,0,2454
3,keywords,405,198.9,22.5,0,118


In [29]:
description_anomalies = df[df['description'].str.split().str.len() >= 1000]
print(f"Indices: {description_anomalies.index.tolist()}\n")

for idx, row in description_anomalies.iterrows():
    print("="*100)
    print(f"LINHA NO DATASET (Indice {idx}):")
    print("="*100)
    print(f"Num of Words : {len(row['description'].split())}\n")
    print(f"• Medical Specialty : {row['medical_specialty']}")
    print(f"• sample_name : {row['sample_name']}") 
    print(f"• description : {str(row['description'])[:150]}") 
    print(f"• keywords : {row['keywords']}") 
    print(f"• transcription : {str(row['transcription'])[:150]}")
    print("\n")
    

Indices: [159, 710, 931, 1058, 2051, 2280]

LINHA NO DATASET (Indice 159):
Num of Words : 1448

• Medical Specialty : Cardiovascular-Pulmonary
• sample_name : psychiatry / psychology, nausea, dizziness, migraine, peripheral neuropathy, diabetic neuropathy, neuropathy, positional vertigo, photophobia and dizziness, neurology consultation, tunnel vision, vestibular migraine, migraine symptoms, headaches, photophobia, ataxia, 
• description : Seizure, hypoglycemia, anemia, dyspnea, edema. colon cancer status post right hemicolectomy, hospital-acquired pneumonia, 
Gastroenterology;Reason for
• keywords : nan
• transcription : nan


LINHA NO DATASET (Indice 710):
Num of Words : 4201

• Medical Specialty : General Medicine
• sample_name :  Pathology - Ovarian Cyst 
• description : Seizure, hypoglycemia, anemia, dyspnea, edema. colon cancer status post right hemicolectomy, hospital-acquired pneumonia, 
Ophthalmology; Excision of 
• keywords : obstetrics / gynecology, right ovarian cyst, speci

In [31]:
descript_anom = [159, 710, 931, 1058, 2051, 2280]
med_specialty_anom = [932, 2052, 2281]

pairing_anom = [(159,0),(710,0),(931,932),(1058,0),(2051,2052),(2280,2281)]

for descript_index, specialty_index in pairing_anom:
    print("="*100)
    print(f"Linha {descript_index} (Description) | Linha {specialty_index} (Medical Specialty)")
    print("="*100)
    # Description Anomalies
    description_text = str(df.loc[descript_index, 'description'])
    print(f"Linha {descript_index} (Description) - Total: {len(description_text)} characters\n")
    print(f" • Primeiros chars: {description_text[:100]}\n")
    print(f" • Ultimos chars: {description_text[-100:]}\n")

    # Medical Specialty Anomalies
    if (specialty_index == 0):
        print("Specialty not related\n")
        continue
    specialty_text = str(df.loc[specialty_index, 'medical_specialty'])
    print(f"Linha {specialty_index} (Med Specialty) - Total: {len(specialty_text)} characters\n")
    print(f" • Primeiros chars: {specialty_text[:100]}\n")
    print(f" • Ultimos chars: {specialty_text[-100:]}\n")    

    print("\n")

    


Linha 159 (Description) | Linha 0 (Medical Specialty)
Linha 159 (Description) - Total: 9722 characters

 • Primeiros chars: Seizure, hypoglycemia, anemia, dyspnea, edema. colon cancer status post right hemicolectomy, hospita

 • Ultimos chars: ouraged her to walk daily in order to improve her fitness, which helps to reduce migraine symptoms."

Specialty not related

Linha 710 (Description) | Linha 0 (Medical Specialty)
Linha 710 (Description) - Total: 28647 characters

 • Primeiros chars: Seizure, hypoglycemia, anemia, dyspnea, edema. colon cancer status post right hemicolectomy, hospita

 • Ultimos chars: 
Obstetrics-Gynecology; Specimen labeled "right ovarian cyst" is received fresh for frozen section."

Specialty not related

Linha 931 (Description) | Linha 932 (Medical Specialty)
Linha 931 (Description) - Total: 32745 characters

 • Primeiros chars: Anterior cervical discectomy, arthrodesis, partial corpectomy, Machine bone allograft, placement of 

 • Ultimos chars: cture wound, a